In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import sys
import html as html_lib
from IPython.display import HTML, display as ipy_display
import matplotlib.pyplot as plt
import pandas as pd
from datasets import load_dataset

root = Path.cwd().resolve()
root = root.parent
sys.path.insert(0, str(root))

import torch
from collections import defaultdict
from transformers import AutoTokenizer, AutoModelForCausalLM
from sae_trainer.core.model_utils import SparseAutoencoder, ReluSparseAutoencoder, TopKSparseAutoencoder
from sae_trainer.sae_semantic_eval.feature_tracer_utils import FeatureTracer, TraceConfig
from sae_trainer.sae_semantic_eval.feature_tracer_eval_utils import (
    compute_coverage_metrics,
    search_features_by_concept, 
    render_feature_card, 
    top_n_feats_by_hits_count, 
    top_n_feats_by_act_mass, 
    prompt_x_feature_heatmap, 
    feature_token_position_profile, 
    top_m_feature_coactivation, 
    render_feature_card, 
    plot_feature_umap, 
    plot_inter_feature_similarity, 
    plot_feature_similarity_violins
)


/Users/alyssagardiner/src/sae_trainer/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
#device = "cuda" if torch.cuda.is_available() else "cpu"
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"device: {device}")
layer_idx = 12
dataset_name = 'openwebtext'
model_name_simple = "qwen" # Either gpt2 or qwen
sae_type = 'topk'
k = 40

if model_name_simple == "gpt2":
    model_name = "gpt2"
elif model_name_simple == "qwen":
    model_name = "Qwen/Qwen2-0.5B-Instruct"
else:
    raise ValueError(f"Invalid model name: {model_name_simple}. Expected either 'gpt2' or 'qwen'.")
tokenizer = AutoTokenizer.from_pretrained(model_name)
llm = AutoModelForCausalLM.from_pretrained(model_name).to(device).eval()

# Load SAE checkpoint — pick whichever layer you want to analyse (3, 6, 9, or 11)
ckpt = torch.load(root / f"model_weights_files/{sae_type}_sae_{model_name_simple}_{dataset_name}_layer{layer_idx}.pt", map_location=device)
if sae_type == 'relu':
    sae = ReluSparseAutoencoder(d_in=ckpt["d_in"], d_latent=ckpt["d_latent"], normalize_decoder=True).to(device)
elif sae_type == 'topk':
    sae = TopKSparseAutoencoder(d_in=ckpt["d_in"], d_latent=ckpt["d_latent"], k=k, normalize_decoder=True).to(device)
else:
    raise ValueError(f"Invalid sae type: {sae_type}. Expected either 'relu' or 'qwen'.")
sae.load_state_dict(ckpt["model_state"])
sae.eval()

device: mps


TopKSparseAutoencoder(
  (encoder): Linear(in_features=896, out_features=7168, bias=True)
  (decoder): Linear(in_features=7168, out_features=896, bias=False)
)

In [4]:
# 4) Tracer config — layer_idx MUST match the checkpoint loaded above
cfg = TraceConfig(
    layer_idx=layer_idx,
    topk_per_token=8,
    min_activation=0.01,
    max_new_tokens=50,
    do_sample=True,
    temperature=0.7,
    token_mode='generated_only',
    min_prompts=5
)
tracer = FeatureTracer(llm, tokenizer, sae, device, cfg)

In [9]:
n_train_docs = 4000 # cfg.max_batches * cfg.collection_batch_size

ds = load_dataset("openwebtext", split="train", streaming=True)
# Exclude training data
ds = ds.skip(n_train_docs)

# Trace once at full budget
tracer = FeatureTracer(llm, tokenizer, sae, device, cfg)
traced_prompts = tracer.trace_prompts_from_iterable_dataset(ds, max_prompts_to_trace=1000)

/Users/alyssagardiner/src/sae_trainer/.venv/lib/python3.12/site-packages/transformers/pytorch_utils.py:325: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_elements = torch.tensor(test_elements)


In [10]:

all_pids = tracer.to_dataframe()["prompt_id"].unique().tolist()

results = []
for n in [100, 250, 500, 750, 1000]:
    sub = tracer.subset(all_pids[:n])
    sub.compute_feature_embeddings()
    sub.feature_specificity_scores()
    metrics = compute_coverage_metrics(sub)
    metrics["n_prompts"] = n
    results.append(metrics)

pd.DataFrame(results)


,n_scored_features,mean_composite_score,mean_context_specificity,frac_above_threshold,split_half_stability,n_prompts
0,331,0.418361,0.018627,0.262840,0.740711,100
1,350,0.267228,0.019511,0.251429,0.825105,250
2,371,0.230243,0.033398,0.385445,0.878924,500
3,381,0.245415,0.030145,0.349081,0.903580,750
4,382,0.194855,0.030214,0.361257,0.919048,1000
